### Imports

In [ ]:

import warnings

import torch
import torch.nn as nn

from src.datasets.audio_dataset import AudioDataset
from src.engine import benchmark_snn, validate_snn, \
    train_one_epoch_snn
from src.models.snn_2d_direct_classifier import SNN2DDirectClassifier
from src.preprocessing import get_snn_pipeline
from src.utils import get_split_dataloaders

warnings.filterwarnings("ignore", category=UserWarning)

### Constants

In [ ]:
INPUT_DIR = '../../data/audioMNIST'
MODEL_PATH = '../../models/best_cnn.pth'

# Hyperparameters
LR = 0.001
NUM_EPOCHS = 20
SLOPE = 25

### Setting up device to use

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

### Training script

In [ ]:
# TODO: Keep track of both MACs and ACs
if __name__ == '__main__':
    dataset = AudioDataset(data_dir=INPUT_DIR, pipeline=get_snn_pipeline())
    train_dataloader, val_dataloader, test_dataloader = get_split_dataloaders(dataset)

    # Get one batch from the training loader and make sure it looks good
    features, labels = next(iter(train_dataloader))
    print(f'Features shape: {features.shape}')
    print(f'Labels shape: {labels.shape}')
    print()

    # Train the model
    model = SNN2DDirectClassifier(slope=SLOPE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # Keep track of best model
    print('Training SNN 2D...')
    best_accuracy = 0.0
    for epoch in range(NUM_EPOCHS):
        saved = False

        print(f'[Epoch {epoch + 1}/{NUM_EPOCHS}]')
        train_loss, train_accuracy = train_one_epoch_snn(device, model, criterion, optimizer, train_dataloader)
        val_loss, val_accuracy = validate_snn(device, model, criterion, val_dataloader)

        if val_accuracy > best_accuracy:
            saved = True
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), MODEL_PATH)

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_accuracy:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_accuracy:.2f}%')
        print()
    print(f'Best model had an accuracy of {best_accuracy:.2f}%.')
    print(f'Running final test...')

    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint, strict=True)
    model.to(device)

    test_accuracy, acs = benchmark_snn(device, model, test_dataloader)
    print(f'Test accuracy: {test_accuracy:.2f}% | Total ACs: {acs}')

Features shape: torch.Size([64, 1, 64, 27])
Labels shape: torch.Size([64])

Training SNN 2D...
[Epoch 1/20]


Validating: 100%|██████████| 47/47 [00:05<00:00,  9.25batches/s]


Train Loss: 1.62 | Train Accuracy: 42.38% | Val Loss: 0.73 | Val Accuracy: 74.77%

[Epoch 2/20]


Validating: 100%|██████████| 47/47 [00:01<00:00, 34.23batches/s]


Train Loss: 0.35 | Train Accuracy: 89.17% | Val Loss: 0.19 | Val Accuracy: 94.33%

[Epoch 3/20]


...

Validating: 100%|██████████| 47/47 [00:01<00:00, 28.33batches/s]


Train Loss: 0.03 | Train Accuracy: 99.38% | Val Loss: 0.04 | Val Accuracy: 98.63%

[Epoch 17/20]


Validating: 100%|██████████| 47/47 [00:01<00:00, 32.64batches/s]


Train Loss: 0.02 | Train Accuracy: 99.46% | Val Loss: 0.03 | Val Accuracy: 99.20%

[Epoch 18/20]


Training:  78%|███████▊  | 292/375 [00:15<00:04, 17.18batches/s]